In [ ]:
import os

from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_core.globals import set_debug, set_verbose
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_community.llms import HuggingFaceHub
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_pinecone import PineconeVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone, ServerlessSpec

set_debug(True)
set_verbose(True)

# Load own documents

In [ ]:
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

In [ ]:
# load own documents from directory

loader = DirectoryLoader("./data/", glob="**/*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

# break down documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
splits = text_splitter.split_documents(docs)

In [ ]:
print("Example content:\n")
print(splits[1].page_content)

print("\nExample metadata:\n")

print(splits[1].metadata)

# Upload to vector db

In [ ]:
# model to generate embeddings

model_name = "BAAI/bge-small-en"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}
embedding_function = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

In [ ]:
# create the vector db / index

index_name = "research-paper-index"

existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]
if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=384,  # based on model output dimensions
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )

In [ ]:
# insert data to vector db
docsearch = PineconeVectorStore.from_documents(
    splits,
    embedding_function,
    index_name=index_name,
)

# view index stats
index = pc.Index(index_name)
index.describe_index_stats()

# docsearch.add_texts(["More text!"])

In [ ]:
# total_vector_count should be same as amounts of chunks

len(splits)

# Retrieve similar data given a query

Perform similarity search to find relevant info in vector db

In [ ]:
index_name = "research-paper-index"

docsearch = PineconeVectorStore(embedding=embedding_function, index_name=index_name)

In [ ]:
query = "Explain saliency detection"
docs = docsearch.similarity_search(query, k=1)
print(docs[0].page_content)

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 1})
matched_docs = retriever.invoke(query)
for i, d in enumerate(matched_docs):
    print(f"\n## Document {i}\n")
    print(d.page_content)

# Generate a response

The result above is not suitable to be output to user. We will need a chat model to reformat the content above to make it more presentable as final result.

In [ ]:
# load open-source chat model from Huggingface
chat_model = HuggingFaceHub(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
    task="text-generation",
    model_kwargs={
        "temperature": 0.001,
        "return_full_text": False,
    },
)


# chat_model = HuggingFaceHub(
#     repo_id="mistralai/Mixtral-8x7B-Instruct-v0.1",
#     task="text-generation",
#     model_kwargs={
#         "temperature": 0.1,
#         "return_full_text" : False
#     },
# )

## Define the output format

In [ ]:
# Define the output data structure


class QuestionAnswer(BaseModel):
    question: str = Field(description="question asked by user")
    answer: str = Field(description="answer from model")

In [ ]:
parser = JsonOutputParser(pydantic_object=QuestionAnswer)
format_instructions = parser.get_format_instructions()

print(format_instructions)

## RAG without own data

In [ ]:
# create the prompt

rag_template_without_context = """ Answer the question based on your understanding. 
Keep the answer short and concise. 
Respond "Unsure about answer" if not sure about the answer.

Question: {question}

{format_instructions}

"""

rag_prompt_without_context = PromptTemplate.from_template(
    template=rag_template_without_context,
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

rag_chain_without_context = (
    {"question": RunnablePassthrough()}
    | rag_prompt_without_context
    | chat_model
    | parser
)

In [ ]:
# run the RAG chain
response_without_context = rag_chain_without_context.invoke(
    "What are the categories of attentional models",
)

In [ ]:
print("Response without own data: \n")
print(response_without_context)

## RAG with own data

In [ ]:
# define the prompt

rag_template_with_context = """ Answer the question based on the context below. 
Keep the answer short and concise. 
Respond "Unsure about answer" if not sure about the answer.

Context: {context}
Question: {question}

{format_instructions}

"""

rag_prompt_with_context = PromptTemplate.from_template(
    template=rag_template_with_context,
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
rag_chain_with_context_from_docs = (
    RunnablePassthrough.assign(context=(lambda x: format_docs(x["context"])))
    | rag_prompt_with_context
    | chat_model
    | parser
)

rag_chain_with_source = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()},
).assign(answer=rag_chain_with_context_from_docs)

In [ ]:
response_with_context_with_source = rag_chain_with_source.invoke(
    "What are the categories of attentional models",
)

In [ ]:
print("Data from db that matched the query:\n")
print(response_with_context_with_source.get("context"))

In [ ]:
print(response_with_context_with_source.get("answer"))